# Milestone 3: Retrieval-Augmented Generation (RAG)

## Setup: install deps, build knowledge base + FAISS index (run first)

In [ ]:
!pip install -q faiss-cpu sentence-transformers transformers scikit-learn

In [ ]:

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline

train = pd.read_csv('train.csv')

print("Creating knowledge base")
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = embed_model.encode(kb, show_progress_bar=False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created. KB size:", len(kb))


## Q1. Zero-shot classification on row 150 (5 options), probability of correct option

In [ ]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])]
ans_150 = str(row_150[row_150['answer']])

result_150 = zs(prompt_150, labels_150)
print(result_150)

idx_of_correct = result_150['labels'].index(ans_150)
prob_correct = result_150['scores'][idx_of_correct]
print("Q1 answer - probability of correct option (rounded to 3):", round(prob_correct, 3))


## Q2. FAISS retrieval rank of the true document for row 150 (top k=10)

In [ ]:
k = 10
query_emb_150 = embed_model.encode([prompt_150])
distances, retrieved_indices = index.search(query_emb_150, k)
retrieved_indices = retrieved_indices[0]
print("Retrieved indices (top 10):", retrieved_indices)

if 150 in retrieved_indices:
    rank_150 = list(retrieved_indices).index(150) + 1
else:
    rank_150 = None  # not found in top 10

print("Q2 answer - rank of true doc (index 150) in FAISS top-10:", rank_150)


## Q3. Cross-Encoder reranking of the top-10 FAISS docs for row 150

In [ ]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

docs_10 = [kb[i] for i in retrieved_indices]
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

sorted_order = np.argsort(ce_scores)[::-1]
sorted_retrieved_indices = [retrieved_indices[i] for i in sorted_order]
print("Cross-encoder sorted KB indices:", sorted_retrieved_indices)

if 150 in sorted_retrieved_indices:
    ce_rank_150 = sorted_retrieved_indices.index(150) + 1
else:
    ce_rank_150 = None

print("Q3 answer - Cross-Encoder rank of true doc:", ce_rank_150)


## Q4. Token count for Context+Question string, row 42, top k=5 docs

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])

k5 = 5
query_emb_42 = embed_model.encode([prompt_42])
_, retrieved_42 = index.search(query_emb_42, k5)
retrieved_42 = retrieved_42[0]

docs_42 = [kb[i] for i in retrieved_42]
concatenated_docs_42 = " ".join(docs_42)

rag_string_42 = f"Context: {concatenated_docs_42} Question: {prompt_42}"

tokens_42 = tokenizer(rag_string_42, truncation=False)
num_tokens_42 = len(tokens_42['input_ids'])

print("Q4 answer - total token count:", num_tokens_42)


## Q5. Zero-shot classification with true document as context (row 150)

In [ ]:
true_doc_150 = kb[150]
rag_string_150 = f"Context: {true_doc_150} Question: {prompt_150}"

result_rag_150 = zs(rag_string_150, labels_150)
print(result_rag_150)

idx_correct_rag = result_rag_150['labels'].index(ans_150)
prob_correct_rag = result_rag_150['scores'][idx_correct_rag]

print("Q5 answer - new probability of correct option with true context (rounded to 3):", round(prob_correct_rag, 3))


## Q6. Adversarial RAG: force context to unrelated KB doc (index 999)

In [ ]:
adversarial_doc = kb[999]
adversarial_string_150 = f"Context: {adversarial_doc} Question: {prompt_150}"

result_adv_150 = zs(adversarial_string_150, labels_150)
print(result_adv_150)

idx_correct_adv = result_adv_150['labels'].index(ans_150)
prob_correct_adv = result_adv_150['scores'][idx_correct_adv]

print("Q6 answer - probability of correct option with adversarial context (rounded to 3):", round(prob_correct_adv, 3))


## Q7. Hit Rate for first 100 rows (top k=5 retrieval)

In [ ]:
k5 = 5
hits = 0
total = 100

for idx in range(total):
    row = train.iloc[idx]
    prompt = str(row['prompt'])
    correct_letter = row['answer']
    correct_text = str(row[correct_letter])

    q_emb = embed_model.encode([prompt])
    _, ret_idx = index.search(q_emb, k5)
    ret_idx = ret_idx[0]

    retrieved_docs = [kb[i] for i in ret_idx]

    if any(correct_text in doc for doc in retrieved_docs):
        hits += 1

hit_rate = (hits / total) * 100
print("Q7 answer - Hit Rate (%):", round(hit_rate, 1))


## Q8. Full RAG pipeline (retrieve -> rerank -> augment -> predict), MAP@3 over rows 0-19

In [ ]:
def map_at_3_single(ranked_labels, correct_label):
    if correct_label in ranked_labels:
        rank = ranked_labels.index(correct_label) + 1
        return 1.0 / rank
    return 0.0

option_letters = ['A', 'B', 'C', 'D', 'E']
k5 = 5
scores = []

for idx in range(20):
    row = train.iloc[idx]
    prompt = str(row['prompt'])
    correct_letter = row['answer']

    # Retrieve top-5
    q_emb = embed_model.encode([prompt])
    _, ret_idx = index.search(q_emb, k5)
    ret_idx = ret_idx[0]
    docs_5 = [kb[i] for i in ret_idx]

    # Rerank with cross-encoder, pick best doc
    pairs_5 = [[prompt, d] for d in docs_5]
    ce_scores_5 = cross_encoder.predict(pairs_5)
    best_doc = docs_5[int(np.argmax(ce_scores_5))]

    # Augment
    rag_string = f"Context: {best_doc} Question: {prompt}"

    # Predict
    labels = [str(row[letter]) for letter in option_letters]
    result = zs(rag_string, labels)

    # Map returned labels (option text) back to letters, sorted by score desc
    label_to_letter = {str(row[letter]): letter for letter in option_letters}
    ranked_letters = [label_to_letter[lab] for lab in result['labels']]
    top3 = ranked_letters[:3]

    score = map_at_3_single(top3, correct_letter)
    scores.append(score)

final_map3 = np.mean(scores)
print("Q8 answer - MAP@3 over 20 rows (rounded to 3):", round(final_map3, 3))
